# Stage 2 constrained-decoding test (Qwen 2.5-7B, fp8, real SKYRAG prompts)

**Upload both files into `work`:** this notebook and `stage2_test_data.json`. Then **Run > Run All Cells**.

What it does:
1. Starts vLLM (Qwen 2.5-7B, fp8, about 11k context, the most this GPU allows).
2. **Stage 1:** refines each benchmark query once with Qwen. The same refined query is reused by every condition, so Stage 1 is held fixed.
3. **Stage 2, four conditions,** on the real 1,000-keyword windows for Coursera and edX, in priority order:
   - `pv`: the flat schema by **prompt-and-validate** (as in the paper; a reply that breaks the schema is discarded)
   - `cd_free`: the same schema **enforced by constrained decoding** (keywords are still free text)
   - `cd_allowed`: constrained decoding **plus** keywords restricted to the window (the allowed list)
   - `arm2`: no schema (the paper's Arm 2 prompt)
4. Scores each condition: invalid rate (ITT and as-treated), silence, violations, keywords emitted. Results are saved after **every** condition.

**This is a pilot on this server only** (fp8, local vLLM). Its numbers must not be mixed with the OpenRouter results.
Download `stage2_results.csv`, `stage2_summary.md` and the logs before the session ends.

In [3]:
# 1. Settings: lower N_QUERIES if you are short of time
N_QUERIES = 25              # of 50; 25 takes roughly 30 to 45 minutes for all four conditions (estimate)
TIME_BUDGET_MIN = 80        # stop starting new conditions after this many minutes
CONDITIONS = ["pv", "cd_free", "cd_allowed", "arm2"]
CONCURRENCY = 2             # this GPU's context memory fits about one or two of these prompts at once

import os, sys, subprocess, time, json, shutil, signal, re, csv, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor

WORK = os.path.expanduser("~/work"); os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
MODEL, PORT = "Qwen/Qwen2.5-7B-Instruct", 8000
VLLM = shutil.which("vllm") or "/opt/conda/bin/vllm"
PY = os.path.join(os.path.dirname(VLLM), "python")
DATA = json.load(open(os.path.join(WORK, "stage2_test_data.json"), encoding="utf-8"))
QUERIES = DATA["queries"][:N_QUERIES]
PLATFORMS = ["coursera", "edx"]
ELIGIBLE = {p: set(DATA["vocab"][p]["eligible"]) for p in PLATFORMS}
WINDOW = {p: DATA["vocab"][p]["window"] for p in PLATFORMS}
MAX_KW = DATA["max_keywords"]
BUDGET = 128 + 32 * MAX_KW      # same completion budget as the harness gives a flat reply (1,408 tokens)
T_START = time.time()
print(len(QUERIES), "queries |", {p: len(WINDOW[p]) for p in PLATFORMS}, "window keywords | budget", BUDGET, "tokens")

25 queries | {'coursera': 1000, 'edx': 1000} window keywords | budget 1408 tokens


In [4]:
# 2. Environment fix (TorchAudio/CUDA mismatch in this image) and GPU check
r = subprocess.run([PY, "-m", "pip", "uninstall", "-y", "torchaudio"], capture_output=True, text=True)
print((r.stdout or r.stderr).strip()[-200:])
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)


name, memory.used [MiB], memory.total [MiB]
NVIDIA GeForce RTX 4070 Ti, 9 MiB, 12282 MiB



In [13]:
# 3. Start the vLLM server
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
def healthy():
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_server(p):
    if p is None:
        return
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM); p.wait(timeout=60)
    except Exception:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception: pass
    time.sleep(5); print("server stopped, GPU freed")

log = open(os.path.join(WORK, "stage2_server.log"), "w")
server = subprocess.Popen([VLLM, "serve", MODEL, "--port", str(PORT), "--quantization", "fp8",
                           "--max-model-len", "11000", "--gpu-memory-utilization", "0.95"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while not healthy():
    if server.poll() is not None:
        raise SystemExit("Server failed to start; read stage2_server.log (last lines) and paste them to Claude.")
    if time.time() - t0 > 45 * 60:
        stop_server(server); raise SystemExit("Server start timed out.")
    time.sleep(5)
print(f"server ready after {(time.time() - t0) / 60:.1f} min")

server ready after 0.6 min


In [7]:
# 4. Request helper
def chat(prompt, max_tokens, schema=None):
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
            "temperature": 0, "seed": 34, "max_tokens": max_tokens}
    if schema is not None:
        body["response_format"] = {"type": "json_schema", "json_schema": {"name": "selection", "schema": schema}}
    req = urllib.request.Request(f"http://localhost:{PORT}/v1/chat/completions",
                                 json.dumps(body).encode(), {"Content-Type": "application/json"})
    t = time.time()
    try:
        r = json.load(urllib.request.urlopen(req, timeout=900))
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}: {e.read().decode(errors='replace')[:400]}", "seconds": time.time() - t}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "seconds": time.time() - t}
    ch = r["choices"][0]
    return {"text": ch["message"]["content"] or "", "finish": ch.get("finish_reason"),
            "completion_tokens": r["usage"]["completion_tokens"], "prompt_tokens": r["usage"]["prompt_tokens"],
            "seconds": time.time() - t}

print("ok")

ok


In [8]:
# 5. Stage 1: refine each query once (reused by every condition)
exp_path = os.path.join(WORK, "stage2_expanded.json")
if os.path.exists(exp_path):
    EXPANDED = json.load(open(exp_path))
else:
    def expand(q):
        r = chat(DATA["expand_prompt"].format(query=q["query"]), 512)
        return q["id"], (r.get("text") or "").strip() or q["query"]
    with ThreadPoolExecutor(4) as ex:
        EXPANDED = dict(ex.map(expand, QUERIES))
    json.dump(EXPANDED, open(exp_path, "w"), indent=1)
print(len(EXPANDED), "expanded; example:\n", EXPANDED[QUERIES[0]["id"]][:400])

25 expanded; example:
 This course is designed for beginners aiming to learn data science within a 6-month period, covering essential topics such as statistical analysis, machine learning algorithms, data visualization, and database management. Participants will gain hands-on experience with Python or R programming languages and tools like SQL, Pandas, and Matplotlib, preparing them to analyze and interpret complex data


In [9]:
# 6. Stage 2: prompts, schemas, validation and scoring
SCHEMA = DATA["schema_v5"]

def allowed_schema(platform):
    s = json.loads(json.dumps(SCHEMA))
    s["properties"]["keywords"]["propertyNames"] = {"enum": WINDOW[platform]}
    s["properties"]["platform"] = {"type": "string", "enum": [platform]}
    return s

def build(cond, platform, query):
    vocab = ", ".join(WINDOW[platform])
    if cond == "arm2":
        return DATA["prompt_arm2"].format(platform=platform, vocabulary=vocab, query=query, max_keywords=MAX_KW), None
    prompt = DATA["prompt_arm3_v5"].format(platform=platform, vocabulary=vocab, query=query,
                                           max_keywords=MAX_KW, schema=DATA["schema_v5_rendered"])
    schema = {"pv": None, "cd_free": SCHEMA, "cd_allowed": allowed_schema(platform)}[cond]
    return prompt, schema

def first_json(text):
    t = re.sub(r"^\s*```(?:json)?|```\s*$", "", text.strip(), flags=re.M).strip()
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(t[a:b + 1])
    except Exception:
        return None

def schema_problem(obj):
    # Mirrors schema_v5: exactly {platform, keywords}; 1 to 40 keywords; scores are numbers in [0, 1].
    if not isinstance(obj, dict): return "not an object"
    if set(obj) != {"platform", "keywords"}: return f"top-level keys {sorted(obj)}"
    if obj["platform"] not in PLATFORMS: return "bad platform"
    kw = obj["keywords"]
    if not isinstance(kw, dict): return "keywords not an object"
    if not 1 <= len(kw) <= MAX_KW: return f"{len(kw)} keywords (allowed 1 to {MAX_KW})"
    for v in kw.values():
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not 0 <= v <= 1:
            return "score out of range"
    return None

def keywords_from(cond, r):
    # Returns (keywords, outcome). An empty list means this platform contributes nothing.
    if "error" in r:
        return [], "request_error"
    if r["finish"] == "length":
        return [], "overrun"
    obj = first_json(r["text"])
    if cond == "arm2":
        if isinstance(obj, dict):
            kw = obj.get("keywords") if isinstance(obj.get("keywords"), dict) else obj
            return [str(k) for k in kw][:MAX_KW], "ok"
        return [], "unparseable"
    if obj is None:
        return [], "violation: unparseable"
    prob = schema_problem(obj)
    if prob:
        return [], f"violation: {prob}"
    return [str(k) for k in obj["keywords"]], "ok"

def is_valid(k):
    return k.strip().lower()

print("ok")

ok


In [10]:
# 7. Run the conditions (results saved after each one)
rows_path = os.path.join(WORK, "stage2_results.csv")
ROWS = []

def run_one(args):
    cond, q, platform = args
    prompt, schema = build(cond, platform, EXPANDED[q["id"]])
    r = chat(prompt, BUDGET, schema)
    kws, outcome = keywords_from(cond, r)
    invalid = [k for k in kws if k.strip().lower() not in ELIGIBLE[platform]]
    return {"condition": cond, "query_id": q["id"], "platform": platform, "outcome": outcome,
            "n_keywords": len(kws), "n_invalid": len(invalid), "invalid_examples": "; ".join(invalid[:5]),
            "finish": r.get("finish"), "prompt_tokens": r.get("prompt_tokens"),
            "completion_tokens": r.get("completion_tokens"), "seconds": round(r.get("seconds", 0), 1),
            "error": r.get("error", ""), "raw": (r.get("text") or "")[:3000]}

for cond in CONDITIONS:
    if (time.time() - T_START) / 60 > TIME_BUDGET_MIN:
        print(f"time budget reached; skipping {cond}")
        continue
    t = time.time()
    jobs = [(cond, q, p) for q in QUERIES for p in PLATFORMS]
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        out = list(ex.map(run_one, jobs))
    ROWS += out
    with open(rows_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(ROWS[0])); w.writeheader(); w.writerows(ROWS)
    bad = sum(r["outcome"] != "ok" for r in out)
    print(f"{cond}: {len(out)} calls in {(time.time() - t) / 60:.1f} min, {bad} not ok")
    errs = [r["error"] for r in out if r["error"]]
    if errs:
        print("   first request error:", errs[0][:300])

pv: 50 calls in 3.0 min, 7 not ok
cd_free: 50 calls in 2.2 min, 0 not ok
cd_allowed: 50 calls in 0.0 min, 50 not ok
   first request error: HTTP 400: {"error":{"message":"Grammar error: Unimplemented keys: [\"propertyNames\"]","type":"BadRequestError","param":null,"code":400}}
arm2: 50 calls in 1.7 min, 0 not ok


In [11]:
# 8. Score: per query, both platforms pooled (as in the paper)
from collections import defaultdict
def summarise(cond):
    rs = [r for r in ROWS if r["condition"] == cond]
    if not rs:
        return None
    byq = defaultdict(list)
    for r in rs:
        byq[r["query_id"]].append(r)
    excluded = [q for q, v in byq.items() if any(x["outcome"] == "request_error" for x in v)]
    itt, at, silent = [], [], 0
    for q, v in byq.items():
        if q in excluded:
            continue
        n = sum(x["n_keywords"] for x in v); bad = sum(x["n_invalid"] for x in v)
        if n == 0:
            silent += 1; itt.append(1.0)
        else:
            itt.append(bad / n); at.append(bad / n)
    calls = [r for r in rs if r["outcome"] != "request_error"]
    return {"condition": cond, "queries": len(itt), "excluded (request errors)": len(excluded),
            "ITT invalid": round(sum(itt) / len(itt), 4) if itt else None,
            "as-treated invalid": round(sum(at) / len(at), 4) if at else None,
            "silent queries": silent,
            "violations": sum(r["outcome"].startswith("violation") for r in calls),
            "overruns": sum(r["outcome"] == "overrun" for r in calls),
            "keywords / call": round(sum(r["n_keywords"] for r in calls) / max(len(calls), 1), 1),
            "sec / call": round(sum(r["seconds"] for r in calls) / max(len(calls), 1), 1)}

S = [s for s in (summarise(c) for c in CONDITIONS) if s]
cols = list(S[0])
table = "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols) + "\n" + "\n".join(
    "| " + " | ".join(str(s[c]) for c in cols) + " |" for s in S)
viol = defaultdict(int)
for r in ROWS:
    if r["outcome"].startswith("violation"):
        viol[(r["condition"], re.sub(r"\d+ keywords", "N keywords", r["outcome"]))] += 1
vtext = "\n".join(f"- {c}: {o} x{n}" for (c, o), n in sorted(viol.items())) or "- none"
note = (f"\n\nPilot on this server: Qwen 2.5-7B, vLLM fp8, max context 11,000, greedy, seed 34, "
        f"{len(QUERIES)} queries x 2 platforms. Not comparable with the OpenRouter results.\n\nViolation reasons:\n{vtext}\n")
open(os.path.join(WORK, "stage2_summary.md"), "w").write(table + note)
print(table + note)

| condition | queries | excluded (request errors) | ITT invalid | as-treated invalid | silent queries | violations | overruns | keywords / call | sec / call |
|---|---|---|---|---|---|---|---|---|---|
| pv | 25 | 0 | 0.4506 | 0.4506 | 0 | 3 | 4 | 16.9 | 7.0 |
| cd_free | 25 | 0 | 0.4895 | 0.4895 | 0 | 0 | 0 | 21.5 | 5.2 |
| cd_allowed | 0 | 25 | None | None | 0 | 0 | 0 | 0.0 | 0.0 |
| arm2 | 25 | 0 | 0.451 | 0.451 | 0 | 0 | 0 | 17.8 | 4.0 |

Pilot on this server: Qwen 2.5-7B, vLLM fp8, max context 11,000, greedy, seed 34, 25 queries x 2 platforms. Not comparable with the OpenRouter results.

Violation reasons:
- pv: violation: N keywords (allowed 1 to 40) x3



In [12]:
# 9. Stop the server
stop_server(server)
print("download:", [f for f in sorted(os.listdir(WORK)) if f.startswith("stage2_")])

server stopped, GPU freed
download: ['stage2_constrained_test.ipynb', 'stage2_expanded.json', 'stage2_results.csv', 'stage2_server.log', 'stage2_summary.md', 'stage2_test_data.json']


In [14]:
# Extra: array payload, free vs allowed-list (the engine does not support propertyNames on objects)
ARR_FREE = {"type": "object", "additionalProperties": False, "required": ["platform", "keywords"],
            "properties": {"platform": {"type": "string", "enum": PLATFORMS},
                           "keywords": {"type": "array", "minItems": 1, "maxItems": MAX_KW,
                                        "items": {"type": "object", "additionalProperties": False,
                                                  "required": ["keyword", "score"],
                                                  "properties": {"keyword": {"type": "string"},
                                                                 "score": {"type": "number", "minimum": 0, "maximum": 1}}}}}}
def arr_allowed(p):
    s = json.loads(json.dumps(ARR_FREE))
    s["properties"]["keywords"]["items"]["properties"]["keyword"] = {"enum": WINDOW[p]}
    s["properties"]["platform"] = {"type": "string", "enum": [p]}
    return s

def build_arr(cond, platform, query):
    p = DATA["prompt_arm3_v5"].format(platform=platform, vocabulary=", ".join(WINDOW[platform]), query=query,
                                      max_keywords=MAX_KW, schema=json.dumps(ARR_FREE, indent=2))
    flat_ex = '{"platform": "%s", "keywords": {"machine learning": 0.95, "statistics": 0.72}}' % platform
    arr_ex = ('{"platform": "%s", "keywords": [{"keyword": "machine learning", "score": 0.95}, '
              '{"keyword": "statistics", "score": 0.72}]}' % platform)
    assert flat_ex in p
    return p.replace(flat_ex, arr_ex), (ARR_FREE if cond == "cd_free_arr" else arr_allowed(platform))

def run_arr(args):
    cond, q, platform = args
    prompt, schema = build_arr(cond, platform, EXPANDED[q["id"]])
    r = chat(prompt, 128 + 64 * MAX_KW, schema)
    if "error" in r:
        kws, outcome = [], "request_error"
    elif r["finish"] == "length":
        kws, outcome = [], "overrun"
    else:
        obj = first_json(r["text"])
        items = obj.get("keywords") if isinstance(obj, dict) else None
        kws = [str(x.get("keyword")) for x in items if isinstance(x, dict)] if isinstance(items, list) else []
        outcome = "ok" if kws else "violation: unparseable or empty"
    invalid = [k for k in kws if k.strip().lower() not in ELIGIBLE[platform]]
    return {"condition": cond, "query_id": q["id"], "platform": platform, "outcome": outcome,
            "n_keywords": len(kws), "n_invalid": len(invalid), "invalid_examples": "; ".join(invalid[:5]),
            "finish": r.get("finish"), "prompt_tokens": r.get("prompt_tokens"),
            "completion_tokens": r.get("completion_tokens"), "seconds": round(r.get("seconds", 0), 1),
            "error": r.get("error", ""), "raw": (r.get("text") or "")[:3000]}

for cond in ["cd_free_arr", "cd_allowed_arr"]:
    t = time.time()
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        out = list(ex.map(run_arr, [(cond, q, p) for q in QUERIES for p in PLATFORMS]))
    ROWS += out
    errs = [r["error"] for r in out if r["error"]]
    print(f"{cond}: {len(out)} calls in {(time.time() - t) / 60:.1f} min, {sum(r['outcome'] != 'ok' for r in out)} not ok",
          ("| first error: " + errs[0][:200]) if errs else "")

with open(rows_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(ROWS[0])); w.writeheader(); w.writerows(ROWS)
S = [s for s in (summarise(c) for c in ["pv", "cd_free", "cd_free_arr", "cd_allowed_arr", "arm2"]) if s]
cols = list(S[0])
table = "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols) + "\n" + "\n".join(
    "| " + " | ".join(str(s[c]) for c in cols) + " |" for s in S)
open(os.path.join(WORK, "stage2_summary.md"), "w").write(table + note)
print(table)
stop_server(server)

cd_free_arr: 50 calls in 2.0 min, 0 not ok 
cd_allowed_arr: 50 calls in 4.5 min, 0 not ok 
| condition | queries | excluded (request errors) | ITT invalid | as-treated invalid | silent queries | violations | overruns | keywords / call | sec / call |
|---|---|---|---|---|---|---|---|---|---|
| pv | 25 | 0 | 0.4506 | 0.4506 | 0 | 3 | 4 | 16.9 | 7.0 |
| cd_free | 25 | 0 | 0.4895 | 0.4895 | 0 | 0 | 0 | 21.5 | 5.2 |
| cd_free_arr | 25 | 0 | 0.455 | 0.455 | 0 | 0 | 0 | 12.6 | 4.7 |
| cd_allowed_arr | 25 | 0 | 0.0 | 0.0 | 0 | 0 | 0 | 31.5 | 10.8 |
| arm2 | 25 | 0 | 0.451 | 0.451 | 0 | 0 | 0 | 17.8 | 4.0 |
server stopped, GPU freed


## Emergency stop

In [ ]:
subprocess.run("pkill -f 'vllm serve'", shell=True); time.sleep(5)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)